```
2. 검수 데이터 추출

NER 태깅 데이터의 주요 오류 패턴을 룰베이스로 탐지하여, 올패스 데이터 중 LLM 검수가 필요한 기사를 1차 추출

step1: 전체 데이터 사전 분석
- 전체 JSON에서 태그, 개체명, 출현빈도, 오류이력, 다중태그를 룰 적용에 필요한 기준 데이터 집계
- 예시: 태그, 개체명, 출현빈도, 개체명ID, 다중태그, 오류빈도 (과거 코드 있음)

step2: 개체명, 문장 단위 룰베이스 검사
- 룰 항목 조건에 따라 Y/N

step3: 기사 단위 결과 집계
- 하나라도 TRUE가 발생하면 해당 기사 전체를 LLM 검수 대상으로 분류

step4: 검수 대상 추출 및 JSON 데이터 생성
```

---

#### 데이터 구조 변환
- json to df

In [2]:
import json
import pandas as pd

json_path = r"C:\work\ner\temp\all_pass_3outlets_1104\강원일보.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for doc in data["document"]:
    doc_meta = doc["metadata"]

    for paragraph in doc["paragraph"]:
        for sentence in paragraph["sentence"]:
            rows.append({
                "id": data["id"],
                "doc_id": doc["id"],
                "sentence_id": sentence["id"],
                "form": sentence["form"]
            })

df = pd.DataFrame(rows)
# df = df.head(50)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 165006 entries, 0 to 165005
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           165006 non-null  str  
 1   doc_id       165006 non-null  str  
 2   sentence_id  165006 non-null  str  
 3   form         165006 non-null  str  
dtypes: str(4)
memory usage: 41.2 MB


In [4]:
df.doc_id.unique()

<ArrowStringArray>
[    'NLRW2600000001.1',     'NLRW2600000001.2',     'NLRW2600000001.5',
     'NLRW2600000001.6',     'NLRW2600000001.7',     'NLRW2600000001.8',
     'NLRW2600000001.9',    'NLRW2600000001.10',    'NLRW2600000001.11',
    'NLRW2600000001.12',
 ...
 'NLRW2600000001.25037', 'NLRW2600000001.25038', 'NLRW2600000001.25039',
 'NLRW2600000001.25040', 'NLRW2600000001.25042', 'NLRW2600000001.25044',
 'NLRW2600000001.25045', 'NLRW2600000001.25047', 'NLRW2600000001.25049',
 'NLRW2600000001.25051']
Length: 14126, dtype: str

### STEP1

- <개체명: 태그> '기존태깅' 추출

In [3]:
# sentence_test = df[df["sentence_id"] == "NLRW2600000001.1.2.1"]

# import re
# re.findall(r"<[^<>:]+:\s*(?:PS|LC|OG|AF|EV|DT)>", sentence_test["form"].iloc[0])

# entity_pattern = r"<[^<>:]+:\s*(?:PS|LC|OG|AF|EV|DT)>"
# df["org_tagging"] = df["form"].apply(lambda text: re.findall(entity_pattern, text))

# df.head(2)

- <개체명: 태그> '개체명', '태그' 추출, '개체명 ID' 생성

In [8]:
pattern = r"<[^<>:]+:\s*(?:PS|LC|OG|AF|EV|DT)>"

# <개체명: 태그> 추출
df["org_tagging"] = df["form"].str.findall(pattern)
df = df.explode("org_tagging", ignore_index=True)
has_entity = df["org_tagging"].notna()

# 개체명 ID 생성
df["entity_id"] = None
df.loc[has_entity, "entity_id"] = (df.loc[has_entity, "sentence_id"] + ".E" + df.loc[has_entity].groupby("sentence_id").cumcount().add(1).astype(str))

# org_tagging 에서 개체명만 추출
df["entity_name"] = df["org_tagging"].str.extract(r"<(.*?):")

# org_tagging 에서 태그만 추출
df["entity_type"] = df["org_tagging"].str.extract(r":\s*(PS|LC|OG|AF|EV|DT)>")

- 태그, 개체명별 출현 빈도 집계
    - 개체명은 띄어쓰기 제거
    - 고빈도: 1, 2, 3, 4 > 1이면 Y
    - 다중태그: 동일 개체명 2개 이상 태그 존재시 Y

In [9]:
# 개체명 내부의 모든 공백 제거
df["temp_entity_name"] = df["entity_name"].str.replace(r"\s+", "", regex=True)

# 태그별 개체명 빈도 집계
freq = (
    df.dropna(subset=["temp_entity_name", "entity_type"])
      .groupby(["entity_type", "temp_entity_name"])
      .size()
      .rename("entity_freq")
      .reset_index()
)

# 추가
df = df.merge(freq, on=["entity_type", "temp_entity_name"], how="left")

# 사분위수
quartile_code = pd.qcut(df["entity_freq"], q=4, labels=False, duplicates="drop")
df["freq_quartile"] = quartile_code.map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4"})

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 359671 entries, 0 to 359670
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                359671 non-null  str    
 1   doc_id            359671 non-null  str    
 2   sentence_id       359671 non-null  str    
 3   form              359671 non-null  str    
 4   org_tagging       315690 non-null  str    
 5   entity_id         315690 non-null  object 
 6   entity_name       315690 non-null  str    
 7   entity_type       315690 non-null  str    
 8   temp_entity_name  315690 non-null  str    
 9   entity_freq       315690 non-null  float64
 10  freq_quartile     315690 non-null  str    
dtypes: float64(1), object(1), str(9)
memory usage: 134.9+ MB


- 오류 목록의 태그, 개체명별 출현 빈도 집계
    - 개체명은 띄어쓰기 제거
    - 고빈도: 1, 2, 3, 4 > 1이면 Y
    - 동일한 태그, 개체명 존재시 Y

In [11]:
import re
import pandas as pd

pattern = r"<[^<>:]+:\s*(?:PS|LC|OG|AF|EV|DT)>"
error_path = r"C:\work\ner\analysis\NER_개체명검수이력_V2.xlsx"

error_df = pd.read_excel(
    error_path,
    sheet_name="오류목록",
    usecols=["개체명", "분류"],
    converters={
        "개체명": lambda value: str(value) if pd.notna(value) else pd.NA,
        "분류": lambda value: str(value).strip() if pd.notna(value) else pd.NA
    }
)

# 태깅분리 = 한 셀에 여러 정답 태깅이 있음(개체명, 태그별로 분리)
error_pairs = []
for _, row in error_df.iterrows():
    pairs = re.findall(pattern, str(row["개체명"]))

    if pairs:
        error_pairs.extend(pairs)
    elif pd.notna(row["개체명"]) and pd.notna(row["분류"]):
        error_pairs.append((row["개체명"], row["분류"]))

error_df_freq = pd.DataFrame(error_pairs, columns=["entity_name", "entity_type"])

# 개체명 내부의 모든 공백 제거
error_df_freq["temp_entity_name"] = error_df_freq["entity_name"].str.replace(r"\s+", "", regex=True)

# 태그별 개체명 빈도 집계
freq = (
    error_df_freq.dropna(subset=["temp_entity_name", "entity_type"])
      .groupby(["entity_type", "temp_entity_name"])
      .size()
      .rename("entity_freq")
      .reset_index()
)

# 추가
error_df_freq = error_df_freq.merge(freq, on=["entity_type", "temp_entity_name"], how="left")

# 사분위수
quartile_code = pd.qcut(error_df_freq["entity_freq"], q=4, labels=False, duplicates="drop")
error_df_freq["freq_quartile"] = quartile_code.map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4"})

# 고빈도로 판단할 사분위수 설정 (만약 갯수가 0이면 Q3, Q2, Q1 순차적으로 추가해줘야함)
target_quartiles = ["Q4","Q3","Q2","Q1"]
error_df_freq["고빈도여부"] = (error_df_freq["freq_quartile"].isin(target_quartiles).map({True: "Y", False: "N"}))

In [12]:
# 실행해서 한번만 가지고 있으면 됨
error_df_freq

,entity_name,entity_type,temp_entity_name,entity_freq,freq_quartile,고빈도여부
0,14일,DT,14일,2,Q1,Y
1,4일,DT,4일,1,Q1,Y
2,AI,AF,AI,4,Q2,Y
3,두 달째,DT,두달째,1,Q1,Y
4,지난 2월,DT,지난2월,1,Q1,Y
...,...,...,...,...,...,...
1108,내년 2월,DT,내년2월,1,Q1,Y
1109,새달,DT,새달,1,Q1,Y
1110,지난해 같은 기간,DT,지난해같은기간,3,Q2,Y
1111,5년,DT,5년,5,Q2,Y


### STEP2

In [3]:
import os
os.getcwd()

'c:\\work\\ner\\src\\data_processing'

In [ ]:
import sys
sys.path.insert(0, r"C:\work\ner")

from src.data_processing.prepare.transform import sentence_data, entity_data
from src.data_processing.prepare.error_stats import build_error_stat

In [ ]:
# ner_data = r"C:\work\ner\temp\all_pass_3outlets_1104\경향신문.json"
error_data = r"C:\work\ner\analysis\NER_개체명검수이력_V2.xlsx"

In [ ]:
# 모듈 수정시 재시작 안하고 불러오기
# import importlib
# importlib.reload(sentence_data)
# importlib.reload(entity_data)
# importlib.reload(build_error_stat)

In [ ]:
# ner_df = entity_data(ner_data)
error_df = build_error_stat(error_data)
# input_data 경로에 저장되고 있음 

In [26]:
# 강원일보
ner_df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 359671 entries, 0 to 359670
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    359671 non-null  str    
 1   doc_id                359671 non-null  str    
 2   sentence_id           359671 non-null  str    
 3   form                  359671 non-null  str    
 4   entity_tagging        315690 non-null  str    
 5   entity_id             315690 non-null  object 
 6   entity_name           315690 non-null  str    
 7   entity_type           315690 non-null  str    
 8   entity_name_no_space  315690 non-null  str    
 9   entity_freq           315690 non-null  float64
 10  freq_quartile         315690 non-null  str    
dtypes: float64(1), object(1), str(9)
memory usage: 134.9+ MB


In [28]:
# 경기일보
ner_df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 571988 entries, 0 to 571987
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    571988 non-null  str    
 1   doc_id                571988 non-null  str    
 2   sentence_id           571988 non-null  str    
 3   form                  571988 non-null  str    
 4   entity_tagging        510394 non-null  str    
 5   entity_id             510394 non-null  object 
 6   entity_name           510394 non-null  str    
 7   entity_type           510394 non-null  str    
 8   entity_name_no_space  510394 non-null  str    
 9   entity_freq           510394 non-null  float64
 10  freq_quartile         510394 non-null  str    
dtypes: float64(1), object(1), str(9)
memory usage: 222.1+ MB


In [29]:
# 경향신문
ner_df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 1360270 entries, 0 to 1360269
Data columns (total 11 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   id                    1360270 non-null  str    
 1   doc_id                1360270 non-null  str    
 2   sentence_id           1360270 non-null  str    
 3   form                  1360270 non-null  str    
 4   entity_tagging        1210840 non-null  str    
 5   entity_id             1210840 non-null  object 
 6   entity_name           1210840 non-null  str    
 7   entity_type           1210840 non-null  str    
 8   entity_name_no_space  1210840 non-null  str    
 9   entity_freq           1210840 non-null  float64
 10  freq_quartile         1210840 non-null  str    
dtypes: float64(1), object(1), str(9)
memory usage: 505.7+ MB


### 문장 단위

In [ ]:
# import pandas as pd
# df1_path = r"C:\work\ner\temp\all_pass_3outlets_1104\강원일보.pkl"
# df1 = pd.read_pickle(df1_path)

In [5]:
df1_path = r"C:\work\ner\temp\all_pass_3outlets_1104\강원일보.json"
df1 = sentence_data(df1_path)

In [4]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 165006 entries, 0 to 165005
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           165006 non-null  str  
 1   doc_id       165006 non-null  str  
 2   sentence_id  165006 non-null  str  
 3   form         165006 non-null  str  
dtypes: str(4)
memory usage: 41.2 MB


#### EXT_S001

In [6]:
df = df1.copy()

In [7]:
# form에 '계엄'이 포함된 행에 EXT_S001 컬럼에 'Y', 포함되지 않은 행에 'N'으로 표시
df["EXT_S001"] = df["form"].str.contains("계엄").map({True: "Y", False: "N"})

#### EXT_S002

In [8]:
# form에 <개체명: 태그> 와 같이 생긴 태깅된 개체명이 하나도 없는 행에 EXT_S002 컬럼에 'Y', 태깅된 개체명이 있는 행에 'N'으로 표시
pattern = r"<[^<>:]+:\s*(?:PS|LC|OG|AF|EV|DT)>"
df["EXT_S002"] = df["form"].str.contains(pattern).map({True: "N", False: "Y"})

#### EXT_S003

In [9]:
# form에 <개체명: DT> 가 연속해서 존재하는 행에 EXT_S003 컬럼에 'Y', 존재하지 않는 행에 'N'으로 표시
pattern = r"<[^<>:]+:\s*DT>(?:\s*<[^<>:]+:\s*DT>)+"
df["EXT_S003"] = df["form"].str.contains(pattern).map({True: "Y", False: "N"})

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 165006 entries, 0 to 165005
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           165006 non-null  str  
 1   doc_id       165006 non-null  str  
 2   sentence_id  165006 non-null  str  
 3   form         165006 non-null  str  
 4   EXT_S001     165006 non-null  str  
 5   EXT_S002     165006 non-null  str  
 6   EXT_S003     165006 non-null  str  
dtypes: str(7)
memory usage: 45.5 MB


### 개체명 단위

In [10]:
entity_df = entity_data(df)

In [11]:
entity_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 359671 entries, 0 to 359670
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    359671 non-null  str    
 1   doc_id                359671 non-null  str    
 2   sentence_id           359671 non-null  str    
 3   form                  359671 non-null  str    
 4   EXT_S001              359671 non-null  str    
 5   EXT_S002              359671 non-null  str    
 6   EXT_S003              359671 non-null  str    
 7   entity_tagging        315690 non-null  str    
 8   entity_id             315690 non-null  object 
 9   entity_name           315690 non-null  str    
 10  entity_type           315690 non-null  str    
 11  entity_name_no_space  315690 non-null  str    
 12  entity_freq           315690 non-null  float64
 13  freq_quartile         315690 non-null  str    
dtypes: float64(1), object(1), str(12)
memory usage: 144.3+ MB


In [40]:
error_df_freq = pd.read_pickle(r"C:\work\ner\analysis\NER_개체명검수이력_V2.pkl")
# dict_df

In [43]:
error_df_freq

,entity_name,entity_type,entity_name_no_space,entity_freq,freq_quartile,고빈도여부
0,14일,DT,14일,2,Q1,Y
1,4일,DT,4일,1,Q1,Y
2,AI,AF,AI,4,Q2,Y
3,두 달째,DT,두달째,1,Q1,Y
4,지난 2월,DT,지난2월,1,Q1,Y
...,...,...,...,...,...,...
1108,내년 2월,DT,내년2월,1,Q1,Y
1109,새달,DT,새달,1,Q1,Y
1110,지난해 같은 기간,DT,지난해같은기간,3,Q2,Y
1111,5년,DT,5년,5,Q2,Y


#### EXT_E001

In [44]:
import numpy as np
error_df = error_df_freq[error_df_freq["고빈도여부"] == "Y"].drop_duplicates(subset=["entity_name_no_space", "entity_type"])[["entity_name_no_space", "entity_type"]].reset_index(drop=True)

# df의 temp_entity_name와 entity_type이 error_df에 존재하는 경우, EXT_E001 컬럼에 'Y', 존재하지 않는 경우 'N'으로 표시
entity_df["EXT_E001"] = np.where(pd.MultiIndex.from_frame(entity_df[["entity_name_no_space", "entity_type"]])
                    .isin(pd.MultiIndex.from_frame(error_df[["entity_name_no_space", "entity_type"]])), "Y", "N")

In [48]:
entity_df["EXT_E001"].value_counts()

EXT_E001
N    326032
Y     33639
Name: count, dtype: int64

#### EXT_E002

In [49]:
# 고빈도로 판단할 사분위수 설정
target_quartiles = ["Q4"]
entity_df["고빈도여부"] = (entity_df["freq_quartile"].isin(target_quartiles).map({True: "Y", False: "N"}))

# 다중태그
tag_count = entity_df.groupby("entity_name_no_space")["entity_type"].transform("nunique")
entity_df["다중태그여부"] = tag_count.ge(2).map({True: "Y", False: "N"})

# 고빈도여부와 다중태그여부가 Y인 경우 EXT_E002 컬럼에 'Y', 그렇지 않은 경우 'N'으로 표시
entity_df["EXT_E002"] = ((entity_df["고빈도여부"] == "Y") & (entity_df["다중태그여부"] == "Y")).map({True: "Y", False: "N"})

#### EXT_E003 : <개체명: 태그> 중 오태깅 사전에 등록된 개체명

In [ ]:
import re
import pandas as pd

dictionary_path = r"C:\work\ner\analysis\NER_오태깅사전.xlsx"

dict_df = pd.read_excel(
    dictionary_path,
    sheet_name="오태깅목록",
    usecols="D,F:G"
)

dict_df.columns = ["pattern", "rule_id", "sub_rule_id"]
# pickle로 저장 

In [26]:
dict_df = dict_df[dict_df["sub_rule_id"] == "OG_001"]

In [34]:
for _, rule in dict_df.iterrows():
    rule_id = rule["rule_id"]
    sub_rule_id = rule["sub_rule_id"]
    pattern = str(rule["pattern"]).strip()

    # 정규식 객체 생성
    regex = re.compile(pattern)

    # 문장 단위 정규식 탐지
    pattern_search = entity_df["form"].fillna("").apply(
        lambda x: [match.group(0) for match in regex.finditer(x)]
    )

    # 개체명 단위 탐지 여부
    entity_df[f"{rule_id}_{sub_rule_id}"] = [
        "Y" if any(entity_name in match for match in matches) else "N"
        for matches, entity_name in zip(pattern_search, entity_df["entity_name"])
    ]

    # 검증용: 실제 탐지 패턴 기록
    entity_df[f"{rule_id}_{sub_rule_id}_PATTERN"] = [
        ", ".join(match for match in matches if entity_name in match)
        for matches, entity_name in zip(pattern_search, entity_df["entity_name"])
    ]

In [ ]:
entity_df[entity_df["EXT_E003_OG_001"] == "Y"][["form","entity_id","entity_tagging","entity_name","EXT_E003_OG_001_PATTERN"]]

In [51]:
entity_df.columns

Index(['id', 'doc_id', 'sentence_id', 'form', 'EXT_S001', 'EXT_S002',
       'EXT_S003', 'entity_tagging', 'entity_id', 'entity_name', 'entity_type',
       'entity_name_no_space', 'entity_freq', 'freq_quartile',
       'EXT_E003_AF_001', 'EXT_E003_AF_002', 'EXT_E003_AF_003',
       'EXT_E003_AF_004', 'EXT_E003_AF_005', 'EXT_E003_AF_006',
       'EXT_E003_AF_007', 'EXT_E003_AF_008', 'EXT_E003_AF_009',
       'EXT_E003_AF_010', 'EXT_E003_DT_001', 'EXT_E003_DT_002',
       'EXT_E003_DT_003', 'EXT_E003_EV_001', 'EXT_E003_EV_002',
       'EXT_E003_LC_001', 'EXT_E003_LC_002', 'EXT_E003_LC_003',
       'EXT_E003_LC_004', 'EXT_E003_LC_005', 'EXT_E003_LC_006',
       'EXT_E003_LC_007', 'EXT_E003_OG_001', 'EXT_E003_OG_002',
       'EXT_E003_OG_003', 'EXT_E003_OG_004', 'EXT_E003_PS_001',
       'EXT_E003_PS_002', 'EXT_E003_PS_003', 'EXT_E003_PS_004',
       'EXT_E003_OG_001_PATTERN', 'EXT_E001', '고빈도여부', '다중태그여부', 'EXT_E002'],
      dtype='str')

##### 내용

| 태그 | 유형명 | 표현 | 정규식 |
|---|---|---|---|
| PS | 직책형 개체명 | ~장 | `r"<[^<>:]+장\s*:\s*PS>"` |
| PS | 영문 단일 문자 | A~Z, A씨, A양 | `r"<[A-Z](?:씨|양)?\s*:\s*PS>"` |
| AF | 법률명 | ~법 | `r"<[^<>:]+법\s*:\s*AF>"` |
| AF | LLM·AI 모델명 | GPT-4o, 딥시크, Claude 등 | `r"<(?:GPT[-\s]?4o|딥시크|엑사원|제미나이3|FraudGPT|WormGPT|동원GPT|나노바나나|클로드|라마|미스트랄|큐웬|그록|젬마|파이|하이퍼클로바X|솔라|카나나|코히어|코파일럿|Claude|Llama|Mistral|Qwen|Grok|Gemma|Phi|HyperCLOVA\s*X|Solar|Kanana|Cohere|Copilot)\s*:\s*AF>"` |
| AF | SNS·커뮤니티 앱 | 인스타그램, 페이스북, 트위터 등 | `r"<(?:인스타그램|페이스북|트위터|엑스|링크드인|텔레그램|카카오톡|스레드|틱톡|스냅챗|레딧|핀터레스트|왓츠앱|디스코드|네이버밴드|카카오스토리)\s*:\s*AF>"` |
| AF | 소프트웨어명 | 포토샵, 엑셀, 크롬 등 | `r"<(?:포토샵|네이버앱|앱스토어|플레이스토어|삼성멤버스|일러스트레이터|프리미어프로|애프터이펙트|엑셀|파워포인트|워드|한글|크롬|엣지|파이어폭스)\s*:\s*AF>"` |
| AF | 상품권·플랫폼명 | ~상품권, ~플랫폼 | `r"<[^<>:]+(?:상품권|플랫폼)\s*:\s*AF>"` |
| DT | 수치 표현 | N번, N년형, N대, N주년 | `r"<\d+\s*(?:번|년형|대|주년)\s*:\s*DT>"` |
| DT | 시간·수량 표현 | ~시간, ~시각, ~명, ~경기 | `r"<[^<>:]+(?:시간|시각|명|경기)\s*:\s*DT>"` |
| EV | 사업·과정 | ~사업, ~과정 | `r"<[^<>:]+(?:사업|과정)\s*:\s*EV>"` |
| LC | 국적·출신 | 	<미국인: LC> 탐지 | r"<(?:미국|한국|중국|일본|영국|독일|프랑스|러시아|캐나다|호주|인도|베트남|태국|필리핀|대만)인\s*:\s*LC>" |
| LC | 국적, 출신 | <미국: LC>인 탐지 | r"<(?:미국|한국|중국|일본|영국|독일|프랑스|러시아|캐나다|호주|인도|베트남|태국|필리핀|대만)\s*:\s*LC>인" |
| OG | 직책 후행 표현 | `<개체명: OG>장`, `<개체명: OG>과장`, `<개체명: OG>소장` | `r"<[^<>:]+\s*:\s*OG>(?:본부장|사령관|과장|관장|소장|장)"` |

In [ ]:
# <개체명: PS>
# ~장
# A,B,...,Z, A씨,B씨,...,Z씨, A양,B양,...,Z양

# <개체명: AF>
# 법률명: ~법
# LLM, AI 모델명: 
# GPT-4o, 딥시크, 엑사원, 제미나이3, FraudGPT, WormGPT, 동원GPT, 나노바나나
# 클로드, 라마, 미스트랄, 큐웬, 그록, 젬마, 파이, 하이퍼클로바X, 솔라, 카나나, 코히어, 코파일럿
# Claude, Llama, Mistral, Qwen, Grok, Gemma, Phi, HyperCLOVA X, Solar, Kanana, Cohere, Copilot

# SNS, 커뮤니티 앱명:
# 인스타그램, 페이스북, 트위터, 엑스, 링크드인, 텔레그램, 카카오톡
# 스레드, 틱톡, 스냅챗, 레딧, 핀터레스트, 왓츠앱, 디스코드, 네이버밴드, 카카오스토리

# 소프트웨어명: 
# 포토샵, 네이버앱, 앱스토어, 플레이스토어, 삼성멤버스
# 일러스트레이터, 프리미어프로, 애프터이펙트, 엑셀, 파워포인트, 워드, 한글, 크롬, 엣지, 파이어폭스

# 상품권, 플랫폼명: ~상품권, ~플랫폼

# <개체명: DT>
# N번
# N년형
# N대
# N주년
# ~시간
# ~시각
# ~명
# ~경기

# <개체명: EV>
# ~사업
# ~과정

# <개체명: LC>
# 미국인, 한국인, 중국인 등 

# <개체명: OG>
# ~장
# ~본부장
# ~과장
# ~사령관
# ~관장
# <개체명: OG>장
# <개체명: OG>과장
# <개체명: OG>소장

#### EXT_E004 : <개체명: DT> 중 구조적 오류가 의심되는 개체명

In [141]:
# import re
# DT_001 = r"<[^<>:]+\s*:\s*DT>\s*(?:쯤|정도|안팎|가량)"
# example_form = "안3<3개월: DT> 정도는 걸린다."
# result = re.findall(DT_001, example_form)
# print(result)

['<3개월: DT> 정도']


In [ ]:
import re
import pandas as pd

dictionary_path = r"C:\work\ner\analysis\NER_오태깅사전.xlsx"

dict_df = pd.read_excel(
    dictionary_path,
    sheet_name="태깅목록",
    usecols="D,F:G"
)

dict_df.columns = ["pattern", "rule_id", "sub_rule_id"]

# 결측치 처리 (이후 계산 때문에)
forms = entity_df["form"]
entity_names = entity_df["entity_name"].fillna("").astype(str)

In [ ]:
# 문장 단위 정규식 탐지
pattern_search = entity_df["form"].fillna("").apply(
    lambda form: [
        match.group(0)
        for match in regex.finditer(form)
    ]
)

# 기존 태깅 문자열이 탐지 패턴에 포함되는지 확인
entity_df[f"{rule_id}_{sub_rule_id}"] = [
    "Y" if any(
        entity_tagging in pattern
        for pattern in patterns
    ) else "N"
    for patterns, entity_tagging in zip(
        pattern_search,
        entity_df["entity_tagging"]
    )
]

# 검증용: 해당 개체명이 포함된 실제 탐지 패턴 기록
entity_df[f"{rule_id}_{sub_rule_id}_PATTERN"] = [
    ", ".join(
        pattern
        for pattern in patterns
        if entity_tagging in pattern
    )
    for patterns, entity_tagging in zip(
        pattern_search,
        entity_df["entity_tagging"]
    )
]

In [184]:
entity_df

,entity_name,tag,org_tagging,탐지 결과,탐지 패턴
0,3개월,DT,<3개월: DT>,N,
1,2026년 11월 6일,DT,<2026년 11월 6일: DT>,Y,<2026년 11월 6일: DT>에서 <7일: DT>까지
2,7일,DT,<7일: DT>,Y,<2026년 11월 6일: DT>에서 <7일: DT>까지


In [ ]:
# import re
# import pandas as pd

# # 테스트 정규식
# DT_004 = r"(?:<[^<>:]+:\s*DT>|[^<>:,]+)\s*(?:부터|에서)\s*(?:(?:<[^<>:]+:\s*DT>|[^<>:,]+)\s*)?(?:까지|사이)|(?:<[^<>:]+:\s*DT>|[^<>:,]+)\s*(?:부터|까지)"

# # 테스트 문장
# example_form = "약 <3개월: DT> 동안 진행됐으며, <2026년 11월 6일: DT>에서 <7일: DT>까지 운영된다."

# # 테스트 개체명 데이터
# entity_data = [
#     {"entity_name": "3개월", "tag": "DT", "org_tagging": "<3개월: DT>"},
#     {"entity_name": "2026년 11월 6일", "tag": "DT", "org_tagging": "<2026년 11월 6일: DT>"},
#     {"entity_name": "7일", "tag": "DT", "org_tagging": "<7일: DT>"},
# ]

# entity_df = pd.DataFrame(entity_data)

# # 정규식 탐지
# regex = re.compile(DT_004)

# pattern_search = [
#     match.group(0)
#     for match in regex.finditer(example_form)
# ]

# print("탐지 패턴:", pattern_search)

# # 개체명별 탐지 결과
# entity_df["탐지 결과"] = [
#     "Y" if any(
#         org_tagging in pattern
#         for pattern in pattern_search
#     ) else "N"
#     for org_tagging in entity_df["org_tagging"]
# ]

# # 실제 탐지 패턴 기록
# entity_df["탐지 패턴"] = [
#     ", ".join(
#         pattern
#         for pattern in pattern_search
#         if org_tagging in pattern
#     )
#     for org_tagging in entity_df["org_tagging"]
# ]

# print(entity_df.to_string(index=False))

탐지 패턴: ['<2026년 11월 6일: DT>에서 <7일: DT>까지']
 entity_name tag        org_tagging 탐지 결과                           탐지 패턴
         3개월  DT          <3개월: DT>     N                                
2026년 11월 6일  DT <2026년 11월 6일: DT>     Y <2026년 11월 6일: DT>에서 <7일: DT>까지
          7일  DT           <7일: DT>     Y <2026년 11월 6일: DT>에서 <7일: DT>까지


#### EXT_E005

In [101]:
tag_data = pd.read_pickle(r"C:\work\ner\analysis\tag_data.pkl")

In [102]:
tag_data

,pattern,rule_id,sub_rule_id
0,(?:약|한|내달|총|지난|오는)\s*<[^<>:]+\s*:\s*DT>,EXT_E004,DT_001
1,<[^<>:]+\s*:\s*DT>\s*(?:쯤|정도|안팎|가량),EXT_E004,DT_002
2,<[^<>:]+\s*:\s*DT>\s*(?:째|때|초|말|경|간|동안),EXT_E004,DT_003
3,"(?:<[^<>:]+:\s*DT>|[^<>:,]+)\s*(?:부터|에서)\s*(?:...",EXT_E004,DT_004
4,<[^<>:]+\s*:\s*DT>\s*(?:내내|만|전|후|뒤|중|내|안|이내)(?...,EXT_E004,DT_005
5,<(?:고생대|중생대|신생대|원생대|고대|중세|근대|현대)\s*:\s*DT>\s*(...,EXT_E004,DT_006
6,<\d+\s*(?:일|월)\s*:\s*DT>\s*(?:날|달),EXT_E004,DT_007
7,"(?<!\d)\d{2,4}",EXT_E005,EV_001
8,"(?<!\d)\d{2,4}\s*(?:년(?:\s*\d{1,2}\s*월)?|시즌)",EXT_E005,EV_002
9,"(?<!\d)\d{2,4}\s*[~-]\s*\d{2,4}",EXT_E005,EV_003


In [87]:
# <개체명: EV> 중 DT 표현 포함 여부 탐지
ext_e005_data = tag_data[tag_data["rule_id"] == "EXT_E005"].reset_index(drop=True)
ev_mask = entity_df["entity_type"].eq("EV")

for _, rule in ext_e005_data.iterrows():
    rule_id = rule["rule_id"]
    sub_rule_id = rule["sub_rule_id"]
    pattern = str(rule["pattern"]).strip()
    regex = re.compile(pattern)

    pattern_search = entity_df["entity_name"].fillna("").apply(
        lambda entity_name: [
            match.group(0)
            for match in regex.finditer(entity_name)
        ]
    )

    result_col = f"{rule_id}_{sub_rule_id}"
    pattern_col = f"{result_col}_PATTERN"

    entity_df[result_col] = "N"
    entity_df[pattern_col] = ""
    entity_df.loc[ev_mask, result_col] = pattern_search.loc[ev_mask].apply(
        lambda matches: "Y" if matches else "N"
    )
    entity_df.loc[ev_mask, pattern_col] = pattern_search.loc[ev_mask].apply(
        lambda matches: ", ".join(matches)
    )


In [ ]:
from IPython.display import display
ext_e005_columns = [
    "sentence_id",
    "entity_id",
    "entity_name",
    "entity_type",
]

for sub_rule_id in ext_e005_data["sub_rule_id"]:
    ext_e005_columns.extend([
        f"EXT_E005_{sub_rule_id}",
        f"EXT_E005_{sub_rule_id}_PATTERN",
    ])


In [90]:
temp = entity_df.loc[entity_df["entity_type"].eq("EV"), ext_e005_columns]

In [ ]:
entity_df[entity_df["entity_id"] == "NLRW2600000001.18677.16.1.E5"]

,id,doc_id,sentence_id,form,EXT_S001,EXT_S002,EXT_S003,entity_tagging,entity_id,entity_name,...,EXT_E005_ EV_001,EXT_E005_ EV_001_PATTERN,EXT_E005_ EV_002,EXT_E005_ EV_002_PATTERN,EXT_E005_ EV_003,EXT_E005_ EV_003_PATTERN,EXT_E005_ EV_004,EXT_E005_ EV_004_PATTERN,EXT_E005_ EV_005,EXT_E005_ EV_005_PATTERN
265413,NLRW2600000001,NLRW2600000001.18677,NLRW2600000001.18677.16.1,<특검팀: OG>은 <이날: DT> 오전 10시 ‘<건진법사: PS>’ <전성배: ...,N,N,N,<2022년 6·1 지방선거: EV>,NLRW2600000001.18677.16.1.E5,2022년 6·1 지방선거,...,Y,2022,Y,2022년,N,,N,,N,


In [100]:
# temp[temp["EXT_E005_  EV_002"] == 'Y']

#### EXT_E006

In [ ]:
# <개체명: EV> 뒤에 이어지는 EV 표현 탐지
ext_e006_data = tag_data[tag_data["rule_id"] == "EXT_E006"].reset_index(drop=True)
all_tag_regex = re.compile(r"<[^<>:]+?\s*:\s*(?:PS|LC|OG|AF|EV|DT)>")

ext_e006_validation = []
for _, rule in ext_e006_data.iterrows():
    rule_id = rule["rule_id"]
    sub_rule_id = rule["sub_rule_id"]
    pattern = str(rule["pattern"]).strip()
    regex = re.compile(pattern)

    result_col = f"{rule_id}_{sub_rule_id}"
    pattern_col = f"{result_col}_PATTERN"
    entity_df[result_col] = "N"
    entity_df[pattern_col] = ""

    checked_count = 0
    matched_count = 0
    mapping_error_count = 0

    for idx in entity_df.index[entity_df["entity_type"].eq("EV")]:
        checked_count += 1
        form_value = entity_df.at[idx, "form"]
        form = "" if pd.isna(form_value) else str(form_value)
        entity_id = str(entity_df.at[idx, "entity_id"])
        entity_order_match = re.search(r"\.E(\d+)$", entity_id)

        if entity_order_match is None:
            mapping_error_count += 1
            continue

        entity_order = int(entity_order_match.group(1)) - 1
        tagged_entities = list(all_tag_regex.finditer(form))

        if entity_order >= len(tagged_entities):
            mapping_error_count += 1
            continue

        target_entity = tagged_entities[entity_order]
        if not re.search(r":\s*EV>$", target_entity.group(0)):
            mapping_error_count += 1
            continue

        target_start = target_entity.start()
        target_end = target_entity.end()
        detected_match = next(
            (
                match
                for match in regex.finditer(form)
                if match.start() <= target_start < target_end <= match.end()
            ),
            None,
        )

        if detected_match:
            matched_count += 1
            entity_df.at[idx, result_col] = "Y"
            entity_df.at[idx, pattern_col] = detected_match.group(0)

    ext_e006_validation.append({
        "sub_rule_id": sub_rule_id,
        "pattern": pattern,
        "regex_compiled": True,
        "EV_checked_count": checked_count,
        "Y_count": matched_count,
        "entity_id_mapping_error_count": mapping_error_count,
    })

ext_e006_validation = pd.DataFrame(ext_e006_validation)
# entity_id와 실제 form상의 태깅 위치를 정확히 매칭하고, 중복 개체명까지 구분하기 위한 검증 로직이 추가됨 

In [104]:
ext_e006_columns = [
    "doc_id",
    "sentence_id",
    "entity_id",
    "entity_name",
    "entity_type",
]

for sub_rule_id in ext_e006_data["sub_rule_id"]:
    ext_e006_columns.extend([
        f"EXT_E006_{sub_rule_id}",
        f"EXT_E006_{sub_rule_id}_PATTERN",
    ])

display(ext_e006_validation)
display(
    entity_df.loc[
        entity_df["entity_type"].eq("EV"),
        ext_e006_columns,
    ]
)


,sub_rule_id,pattern,regex_compiled,EV_checked_count,Y_count,entity_id_mapping_error_count
0,EV_001,<[^<>:]+\s*:\s*EV>\s*때,True,12374,36,0
1,EV_002,"<[^<>:]+\s*:\s*EV>\s*\d{1,2}\s*(?:라운드|회|세트|경기|...",True,12374,63,0


,doc_id,sentence_id,entity_id,entity_name,entity_type,EXT_E006_EV_001,EXT_E006_EV_001_PATTERN,EXT_E006_EV_002,EXT_E006_EV_002_PATTERN
243,NLRW2600000001.14,NLRW2600000001.14.6.1,NLRW2600000001.14.6.1.E3,인제군민과 함께하는 스케이트 교실,EV,N,,N,
307,NLRW2600000001.23,NLRW2600000001.23.6.1,NLRW2600000001.23.6.1.E3,12·3 내란 사태,EV,N,,N,
404,NLRW2600000001.36,NLRW2600000001.36.4.1,NLRW2600000001.36.4.1.E5,UN기후변화협약,EV,N,,N,
632,NLRW2600000001.52,NLRW2600000001.52.7.1,NLRW2600000001.52.7.1.E4,카라바오컵,EV,N,,N,
660,NLRW2600000001.52,NLRW2600000001.52.9.1,NLRW2600000001.52.9.1.E5,FA컵,EV,N,,Y,<FA컵: EV> 3라운드
...,...,...,...,...,...,...,...,...,...
359435,NLRW2600000001.25031,NLRW2600000001.25031.8.1,NLRW2600000001.25031.8.1.E2,한국감사인대회,EV,N,,N,
359529,NLRW2600000001.25039,NLRW2600000001.25039.6.1,NLRW2600000001.25039.6.1.E3,코리아그랜드페스티벌,EV,N,,N,
359626,NLRW2600000001.25049,NLRW2600000001.25049.7.1,NLRW2600000001.25049.7.1.E2,빛의 바다 속초,EV,N,,N,
359645,NLRW2600000001.25049,NLRW2600000001.25049.11.1,NLRW2600000001.25049.11.1.E6,속초시민 새해맞이 행사,EV,N,,N,


#### EXT_E007

In [ ]:
# 태깅 사전에 존재하는 DT 표현이 form에 포함되어 있고 entity_id 와 함께 확인시 탐지패턴에 기존태깅이 포함되어 있지 않으면 Y (이건 문장단위로 Y를 찍어야할지 개체단위로 Y를 찍어야할지?)

#### 내용

태깅 표현
| 유형명 | 표현 | 정규식 |
|---|---|---|
| 상대 기간 | 하루, 이틀, 주말, 일주일 | `(?:하루|이틀|주말|일주일)` |
| 월·연 단위 | 월초, 월말, 연초, 연말, 연휴 | `(?:월초|월말|연초|연말|연휴)` |
| 기간 구분 | 전반기, 후반기, 성수기, 비수기, 환절기 | `(?:전반기|후반기|성수기|비수기|환절기)` |
| 요일 구분 | 평일, 공휴일 | `(?:평일|공휴일)` |
| 월 상대 표현 | 이달, 지난달, 다음달 | `(?:이달|지난달|다음달)` |
| 주 상대 표현 | 지난주, 이번주, 다음주 | `(?:지난주|이번주|다음주)` |
| 주차 | 첫주, 둘째주 등 | `(?:첫|둘째|셋째|넷째|다섯째|마지막)\s*주` |
| 요일 전체명 | 월요일~일요일 | `(?:월|화|수|목|금|토|일)요일` |
| 요일 약칭 | 월, 화, 수, 목, 금, 토, 일 | `(?:월|화|수|목|금|토|일)` |
| 수치 기간 | N일, N월, N주, N개월, N년 | `\d+\s*(?:일|월|주|개월|년)` |
| 숙박 기간 | N박 N일 | `\d+\s*박\s*\d+\s*일` |
| 연월일 | N년 N월 N일 | `\d{2,4}\s*년\s*\d{1,2}\s*월\s*\d{1,2}\s*일` |
| 연월 | N년 N월 | `\d{2,4}\s*년\s*\d{1,2}\s*월` |
| 월일 | N월 N일 | `\d{1,2}\s*월\s*\d{1,2}\s*일` |
| 계절 | 봄, 여름, 가을, 겨울 및 철·계 | `(?:봄|여름|가을|겨울)(?:철)?|(?:춘계|하계|추계|동계)` |
| 절기 | 입춘~대한 | `(?:입춘|우수|경칩|춘분|청명|곡우|입하|소만|망종|하지|소서|대서|입추|처서|백로|추분|한로|상강|입동|소설|대설|동지|소한|대한)` |
| 기념일·명절 | 명절, 설날, 근로자의 날 등 | `(?:명절|설날|근로자의\s*날|어버이날|빼빼로데이|크리스마스(?:이브)?|가정의\s*달|호국보훈의\s*달)` |
| 상대 날짜 | 어제, 오늘, 금일, 당일, 내일, 모레 | `(?:어제|오늘|금일|당일|내일|모레|첫날|마지막날)` |
| 유행 표현 | 불금, 놀토, 불토 | `(?:불금|놀토|불토)` |
| 연대·출생 | N년대, N년경, BC N년, N년생 | `(?:BC\s*)?\d+\s*년(?:대|경|생)` |
| 연도 상대 표현 | 작년, 올해, 내년 등 | `(?:작년|올해|내년|새해|첫해|마지막해)` |
| 지질시대 | 캄브리아기, 중생대 등 | `(?:캄브리아기|중생대|원생대|올리고세|에오세|신생대|선사시대|석탄기|빙기|고생대|구석기시대|신석기시대|청동기시대|철기시대)` |
| 육십갑자 | 갑자년~계해년 | `(?:갑자|을축|병인|정묘|무진|기사|경오|신미|임신|계유|갑술|을해|병자|정축|무인|기묘|경진|신사|임오|계미|갑신|을유|병술|정해|무자|기축|경인|신묘|임진|계사|갑오|을미|병신|정유|무술|기해|경자|신축|임인|계묘|갑진|을사|병오|정미|무신|기유|경술|신해|임자|계축|갑인|을묘|병진|정사|무오|기미|경신|신유|임술|계해)년` |
| 십이간지 | 쥐띠 해~돼지띠 해 | `(?:쥐|소|호랑이|토끼|용|뱀|말|양|원숭이|닭|개|돼지)띠\s*해` |
| 연호 | 건양·광무·메이지·레이와 등 | `(?:건양|광무|융희|홍무|영락|선덕|정통|성화|홍치|가정|만력|천계|숭정|강희|옹정|건륭|가경|도광|함풍|동치|광서|선통|메이지|다이쇼|쇼와|헤이세이|레이와)\s*\d*\s*년?` |
| 왕조·통치시대 | 정조 시대, 황제 시대 | `(?:정조|가리에누스\s*황제)\s*시대` |
| 역사시대 | 고대, 중세, 조선시대 등 | `(?:고대|중세|조선|가리에누스\s*황제\s*시대|고구려시대|고려시대|조선시대|청대|당대|구한말|일제강점기)` |

문맥 확인이 필요한 표현

| 유형명 | 표현 | 정규식 |
|---|---|---|
| 수치 일수 | N일, N일째, N일차 | `\d+\s*일(?:째|차)?` |
| 방학 | 방학 | `방학` |
| 학기 | N학기 | `\d+\s*학기` |

태깅 제외 표현

| 유형명 | 표현 | 정규식 |
|---|---|---|
| 반복 기간 | 매일, 매주, 매달, 매월, 매해, 매년 | `(?:매일|매주|매달|매월|매해|매년)` |
| 현재 시대 | 현대 | `현대` |

DT 범위 표현

| 유형명 | 표현 | 정규식 |
|---|---|---|
| 선행 표현 | 약~, 한~, 내달~, 총~, 지난~, 오는~ | `(?:약|한|내달|총|지난|오는)\s*` |
| 근사 범위 | ~여, ~쯤, ~정도, ~안팎, ~가량 | `(?:여|쯤|정도|안팎|가량)` |
| 시점·기간 후행 | ~째, ~때, ~초, ~말, ~경, ~간, ~동안 | `(?:째|때|초|말|경|간|동안)` |
| 기간 범위 | ~부터, ~까지, ~에서 ~까지, ~에서 ~사이 | `(?:부터|까지|에서\s*.*?까지|에서\s*.*?사이)` |
| 기간 내부 | ~내내, ~만, ~씩, ~전, ~후, ~뒤, ~중, ~내, ~안, ~이내 | `(?:내내|만|씩|전|후|뒤|중|내|안|이내)` |
| 시즌·연·세기 | ~시즌, 한 해, 세기 | `(?:시즌|한\s*해|세기)` |
| 일·날 결합 | N일 날 | `\d+\s*일\s*날` |
| 월·달 결합 | N월 달 | `\d+\s*월\s*달` |
| 시대·기 결합 | 고생대 오르도비스기 등 | `(?:고생대|중생대|신생대|원생대|고대|중세|근대|현대)\s+\S+기` |

In [ ]:
# [태깅 표현]

# 하루, 이틀, 주말, 일주일
# 월초, 월말, 연초, 연말, 연휴
# 전반기, 후반기, 성수기, 비수기, 환절기

# 평일, 공휴일
# 이달, 지난달, 다음달
# 지난주, 이번주, 다음주 ...
# 첫주, 둘째주 ...

# 월요일, 화요일, 수요일, 목요일, 금요일, 토요일, 일요일 
# 월, 화, 수, 목, 금, 토, 일

# N일, N월, N주, N개월, N년, N박 N일
# 년+월+일, 년+월, 월+일

# 봄, 봄철, 춘계
# 입춘, 우수, 경칩, 춘분, 청명, 곡우
# 여름, 여름철, 하계
# 입하, 소만, 망종, 하지, 소서, 대서
# 가을, 가을철, 추계
# 입추, 처서, 백로, 추분, 한로, 상강
# 겨울, 겨울철, 동계
# 입동, 소설, 대설, 동지, 소한, 대한

# 명절, 설날, 근로자의 날, 어버이날, 빼빼로데이, 크리스마스, 크리스마스이브, 가정의 달, 호국보훈의 달
# 어제, 오늘, 금일, 당일, 내일, 모레, 첫날, 마지막날
# 불금, 놀토, 불토

# N년대, N년경, BC N년, N년생
# 작년, 올해, 내년, 새해, 첫해, 마지막해

# 지질시대:
# 캄브리아기, 중생대, 원생대, 올리고세, 에오세, 신생대, 선사시대, 석탄기, 빙기, 고생대
# 구석기시대, 신석기시대, 청동기시대, 철기시대

# 육십갑자+년:
# 갑자년, 을축년, 병인년, 정묘년, 무진년, 기사년, 경오년, 신미년, 임신년, 계유년,
# 갑술년, 을해년, 병자년, 정축년, 무인년, 기묘년, 경진년, 신사년, 임오년, 계미년,
# 갑신년, 을유년, 병술년, 정해년, 무자년, 기축년, 경인년, 신묘년, 임진년, 계사년,
# 갑오년, 을미년, 병신년, 정유년, 무술년, 기해년, 경자년, 신축년, 임인년, 계묘년,
# 갑진년, 을사년, 병오년, 정미년, 무신년, 기유년, 경술년, 신해년, 임자년, 계축년,
# 갑인년, 을묘년, 병진년, 정사년, 무오년, 기미년, 경신년, 신유년, 임술년, 계해년

# 십이간지+해:
# 쥐띠 해, 소띠 해, 호랑이띠 해, 토끼띠 해
# 용띠 해, 뱀띠 해, 말띠 해, 양띠 해
# 원숭이띠 해, 닭띠 해, 개띠 해, 돼지띠 해

# 연호+연도:
# 한국: 건양, 광무, 융희 N년
# 중국: 홍무, 영락, 선덕, 정통, 성화, 홍치, 가정, 만력, 천계, 숭정, 강희, 옹정, 건륭, 가경, 도광, 함풍, 동치, 광서, 선통
# 일본: 메이지, 다이쇼, 쇼와, 헤이세이, 레이와

# 왕조:
# 정조 시대, 가리에누스 황제 시대

# 역사적 시대 명칭
# 고대, 중세, 조선
# 가리에누스 황제 시대, 고구려시대, 고려시대, 조선시대, 청대, 당대 등
# 구한말, 일제강점기


In [ ]:
# [문맥 확인이 필요한 태깅 표현]
# N일, N일째, N일차
# 방학
# N학기

In [ ]:
# [태깅 제외 표현]
# 매일, 매주, 매달, 매월, 매해, 매년, 현대

In [ ]:
# [DT 범위 표현]

# 선행표현:
# 약~, 한~, 내달~, 총~, 지난~, 오는~, 

# 후행표현:
# ~여, ~쯤, ~정도, ~안팎, ~째, ~때, ~초, ~말, ~경, ~간, ~내내, ~동안, ~가량, ~만, ~씩, ~전, ~후, ~뒤, ~중, ~내, ~안, ~이내
# ~부터 ~까지, ~에서 ~까지, ~에서 ~사이, ~부터, ~까지
# ~시즌, ~한 해, ~세기

# N주 + 요일은 각각 태깅 <셋째 주: DT> <수요일: DT>
# 연/월/일 + 요일 에서 요일은 각각 태깅 

# n일 + 날은 전체 태깅 <10일 날: DT>
# n월 + 달은 전체 태깅 <10월 달: DT>
# 시대 + 기(period) 는 전체 태깅 <고생대 오르도비스기: DT>, <중생대 초기: DT_GEOAGE>
# EV + 때는 DT로 전체 태깅 <제2차 세계대전 때: DT>

### STEP3

### STEP4
- 저장

In [ ]:
import json
import pandas as pd

cols = [
    'id', 'doc_id', 'sentence_id',
    'entity_id', 'form',
    'temp_entity_name', 'entity_name', 'entity_type',
    'EXT_S001', 'EXT_S002', 'EXT_S003',
    'EXT_E001', 'EXT_E002'
]

entity_cols = [
    'id', 'entity_id',
    'temp_entity_name', 'entity_name', 'entity_type',
    'EXT_S001', 'EXT_S002', 'EXT_S003',
    'EXT_E001', 'EXT_E002'
]

review_df = df[cols].copy()

# NaN → None
review_df = review_df.astype(object).where(
    pd.notna(review_df),
    None
)

with open('review_data.jsonl', 'w', encoding='utf-8') as f:
    for (doc_id, sentence_id), group in review_df.groupby(
        ['doc_id', 'sentence_id'],
        sort=False,
        dropna=False
    ):
        form_values = group['form'].dropna().unique()

        if len(form_values) > 1:
            raise ValueError(
                f'{sentence_id}에 여러 form 값이 있습니다: {form_values}'
            )

        record = {
            'doc_id': doc_id,
            'sentence_id': sentence_id,
            'form': form_values[0] if len(form_values) else None,
            'entities': group[entity_cols].to_dict(
                orient='records'
            )
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
                separators=(',', ':')
            )
            + '\n'
        )

# sentences
# - doc_id
# - sentence_id
# - sentence

# entities
# - entity_id
# - sentence_id
# - form
# - entity_name
# - entity_type

# entity_ext
# - entity_id
# - EXT_S001
# - EXT_S002
# - EXT_S003
# - EXT_E001